# Function 4
This function has 4 dimensions.

In [2]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [3]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.68348638 0.9027701  0.33541983 0.99948256]
 [0.17034731 

Here we add the data provided with the weekly queries

In [4]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5], [0.005543, 0.364356, 0.999366, 0.996013], [0.610283, 0.99932, 0.996058, 0.008603], [0.002628, 0.992778, 0.996738, 0.324661]]
additionalOutputs = [np.float64(-3.985750969810286), np.float64(-35.05101840342992), np.float64(-36.45058526638413), np.float64(-34.837035799105976)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.68348638 0.9027701  0.33541983 0.99948256]
 [0.17034731 

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [5]:
#Initialise query lists and maximum observations
X, Y = input, output

#Initialise grid for plots
vector_values = np.linspace(0, 1, 101)# 101 values from 0 to 1
#print(vector_values)

# --- Latin Hypercube Sampling (LHS) ---
n_samples = 12500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(12500000, 4)


In [6]:
print(x_grid[:10])

[[0.07564682 0.19730556 0.61438713 0.9059265 ]
 [0.29252671 0.03016936 0.6251265  0.07986546]
 [0.79961855 0.61025564 0.27169621 0.86352841]
 [0.94375003 0.14954025 0.35439468 0.42597542]
 [0.00237004 0.12494751 0.09195369 0.27859139]
 [0.93379658 0.02449221 0.97306184 0.30644521]
 [0.05330282 0.53045086 0.98159828 0.84677248]
 [0.32496575 0.80649195 0.67306674 0.55027   ]
 [0.18394533 0.21741149 0.4257746  0.58111678]
 [0.76197927 0.99845388 0.94852926 0.00507875]]


# Bayesian Optimisation with UCB applied

In [7]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
beta = 1.96
#beta = 0.5

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.97426599 0.00135303 0.99799554 0.9929418 ]
